In [1]:
import src.database.scripts.sql as sql
from src.database.scrappers.recipes_fetch import merchant_recipes

In [2]:
def drop_recipes_table():
    query = "DROP TABLE IF EXISTS season_8.recipes"
    cursor.execute(query)

def create_recipes_table():
    query = """
    CREATE TABLE IF NOT EXISTS recipes(
    recipe_id   INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    name        TEXT,
    amount      INT,
    rarity      TEXT,
    merchant    TEXT,
    affinity    INT
    )"""
    cursor.execute(query)

def recipes_fetch():
    recipes_list = []
    for merchant in merchant_recipes.merchant_list:
        print(f'parsing {merchant}')
        table = merchant_recipes(merchant)
        for row in table.row_range():
            table.row_target(row)
            amount, rarity, name = table.item()
            merchant, affinity = table.vendor()
            recipes_list.append((name, amount, rarity, merchant, affinity))
    return recipes_list

def insert_recipes_table():
    row_list = recipes_fetch() 
    query = """
    INSERT INTO season_8.recipes (name, amount, rarity, merchant, affinity)
    VALUES (%s, %s, %s, %s, %s)
    """
    cursor.executemany(query, row_list)

In [4]:
def drop_ingredients_table():
    query = "DROP TABLE IF EXISTS season_8.ingredients"
    cursor.execute(query)

def create_ingredients_table():
    query = """
    CREATE TABLE IF NOT EXISTS ingredients(
    recipe_id   INT NOT NULL REFERENCES season_8.recipes(recipe_id),
    name        TEXT,
    amount      INT,  
    rarity      TEXT,
    PRIMARY KEY (recipe_id, name, rarity)
    )"""
    cursor.execute(query)

def ingredients_fetch():
    ingredients_list = []
    recipe_id = 1
    for merchant in merchant_recipes.merchant_list:
        print(f'parsing {merchant}')
        table = merchant_recipes(merchant)
        for row in table.row_range():
            table.row_target(row)
            ingredients = table.ingredients()
            ingredients = [(recipe_id,) + t for t in ingredients]
            ingredients_list.extend(ingredients)
            recipe_id += 1
    return ingredients_list

def insert_ingredients_table():
    ingredients_list = ingredients_fetch()
    query = """
    INSERT INTO season_8.ingredients (recipe_id, amount, rarity, name)
    VALUES (%s, %s, %s, %s)
    """
    cursor.executemany(query, ingredients_list)

def amount_correction():
    query = """
    UPDATE season_8.recipes
    SET amount = 3
    WHERE name LIKE '%Potion%'
    OR name = 'Poison Vial'
    OR name = 'Ghostdust Pouch'
    """
    cursor.execute(query)

In [4]:
def recipe_creation():
    if __name__ == "__main__":
        conn = sql.connect_pc()
        cursor = conn.cursor()

        drop_recipes_table()
        create_recipes_table()
        insert_recipes_table()

        conn.commit()
        conn.close()

def ingredients_creation():
    if __name__ == "__main__":
        conn = sql.connect_pc()
        cursor = conn.cursor()

        drop_ingredients_table()
        create_ingredients_table()
        ingredients_fetch()
        insert_ingredients_table()
        amount_correction()

        conn.commit()
        conn.close()

In [5]:
if __name__ == "__main__":
    conn = sql.connect_pc()
    cursor = conn.cursor()

    # print('dropping table...')
    # drop_ingredients_table()
    # print('creating table...')    
    # create_ingredients_table()
    # print('inserting ingredients rows...')
    # insert_ingredients_table()
    print('correcting amount...')
    amount_correction()

    conn.commit()
    conn.close()

correcting amount...
